# Exercise 11 - Cython and Numba

## Cython

In [1]:
import Cython
%load_ext Cython

# Note: To use Cython on a Windows machine you may need to install Visual Studio to use it 
# to install the Python Extensions Package

#### Create random array
https://docs.python.org/3/library/array.html

In [2]:
from array import array
from random import randint

In [20]:
# create random array
arr = array('i', (randint(-1000, 1000) for _ in range(10000000)))

In [4]:
def sum_py(arr):
    sum_ = 0
    for elem in arr:
        sum_ += elem
    return sum_

In [5]:
%timeit -r3 -n5 sum(arr)
%timeit -r3 -n5 sum_py(arr)

287 ms ± 20.2 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)
679 ms ± 12.5 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)


In [8]:
%%cython
def sum_cy(arr):
    sum_ = 0
    for elem in arr:
        sum_ += elem
    return sum_

In [9]:
%timeit -r3 -n5 sum(arr)
%timeit -r3 -n5 sum_cy(arr)

372 ms ± 3.37 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)
616 ms ± 11.6 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)


#### Static Typing

In [10]:
%%cython
def sum_cy_st(arr):
    cdef int sum_ = 0
    for elem in arr:
        sum_ += elem
    return sum_

In [11]:
%timeit -r3 -n5 sum(arr)
%timeit -r3 -n5 sum_cy_st(arr)

359 ms ± 9.62 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)
853 ms ± 17.1 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)


#### Typed Memoryviews

In [12]:
%%cython
def sum_cy_tm_1(int[:] arr):
    cdef int sum_ = 0
    for elem in arr:
        sum_ += elem
    return sum_

In [13]:
%timeit -r3 -n5 sum(arr)
%timeit -r3 -n5 sum_cy_st(arr)
%timeit -r3 -n5 sum_cy_tm_1(arr)

360 ms ± 7.45 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)
896 ms ± 25.3 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)
3.89 s ± 16.5 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)


#### With C-like looping

In [35]:
%%cython
from cpython cimport array
def sum_cy_tm_2(int[:] arr):
    cdef int i, n = arr.shape[0], sum_ = 0
    for i in range(n):
        sum_ += arr[i]
    return sum_

In [36]:
#%timeit -r3 -n5 sum(arr)
#%timeit -r3 -n5 sum_cy_st(arr)
%timeit -r1 -n1 sum_cy_tm_2(arr)

42.3 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


#### Compile Optimizations

In [37]:
%%cython
cimport cython
from cpython cimport array

@cython.boundscheck(False)  # just read from/write to memory without checking if an index is in a given range
@cython.wraparound(False)  # using negative indices is not possible anymore
# see also: https://docs.cython.org/en/latest/src/userguide/source_files_and_compilation.html#compiler-directives
def sum_cy_co(int[::1] arr):
    cdef int i, n = arr.shape[0], sum_ = 0
    for i in range(n):
        sum_ += arr[i]
    return sum_

In [33]:
%timeit -r3 -n5 sum(arr)
%timeit -r3 -n5 sum_cy_co(arr)

430 ms ± 8.04 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)
12.2 ms ± 1.68 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)


In [39]:
# With Memory view layout --> int[::1] when passing an array as function argument
# see also: https://cython.readthedocs.io/en/latest/src/userguide/memoryviews.html
%timeit -r3 -n5 sum(arr)
%timeit -r3 -n5 sum_cy_co(arr)

362 ms ± 1.97 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)
7.79 ms ± 991 µs per loop (mean ± std. dev. of 3 runs, 5 loops each)


#### Together with Multithreading

In [43]:
%%cython
cimport cython
from cpython cimport array

@cython.boundscheck(False)
@cython.wraparound(False)
def _sum_gil(int[::1] arr, int[:] out):
    cdef int i, n = arr.shape[0], sum_ = 0
    
    with nogil:
        for i in range(n):
            sum_ += arr[i]
        
    out[0] = sum_
      
def sum_threads(int[::1] arr):
    cdef int[::1] arr1 = arr[0 : arr.shape[0]//2]
    cdef int[::1] arr2 = arr[arr.shape[0]//2 : arr.shape[0]]
    cdef int[:] out = array.array('i', (0, 0))
    
    from threading import Thread
    t1 = Thread(target=_sum_gil, args=(arr1, out[0:1]))
    t2 = Thread(target=_sum_gil, args=(arr2, out[1:2]))
    
    t1.start(); t2.start()
    t1.join(); t2.join()
    
    return out[0] + out[1]

In [42]:
# with GIL
%timeit -r3 -n5 sum(arr)
%timeit -r3 -n5 sum_threads(arr)

364 ms ± 1.75 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)
10.2 ms ± 648 µs per loop (mean ± std. dev. of 3 runs, 5 loops each)


In [44]:
# with GIL released (use with nogil statement)
%timeit -r3 -n5 sum(arr)
%timeit -r3 -n5 sum_threads(arr)

374 ms ± 7.78 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)
7.77 ms ± 1.37 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)


#### Interfacing with C code
after -I argument we need to specify the path where sum.h is located

In [ ]:
%%cython -I.
cimport cython
from cpython cimport array

cdef extern from "sum.h" nogil:
    cdef int sum_c(int* a, int n)
    
@cython.boundscheck(False)
@cython.wraparound(False)
def sum_cy_c(int[::1] a):
    cdef int out
    with nogil: 
        out = sum_c(&a[0], a.shape[0])
    return out

#### Numpy (Vectorization)

In [46]:
import numpy as np

np_arr = np.asarray(arr)

%timeit -r3 -n5 sum(np_arr)
%timeit -r3 -n5 np_arr.sum()
%timeit -r3 -n5 sum_cy_co(np_arr)

692 ms ± 66.8 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)
22.9 ms ± 10.2 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)
15.7 ms ± 3.43 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)


#### Proof that they are all working

In [47]:
(sum_py(arr), sum_cy(arr), sum(arr), sum_cy_st(arr), sum_cy_tm_1(arr), sum_cy_tm_2(arr), sum_cy_co(arr),
 sum_threads(arr), np_arr.sum())

(794807, 794807, 794807, 794807, 794807, 794807, 794807, 794807, 794807)

## Numba

In [48]:
import os
os.environ['NUMBA_NUM_THREADS'] = '10'
import numba as nb
import numpy as np

In [49]:
arr = np.random.random((4096,4096))

In [51]:
def py_sum2d(arr):
    result = 0.0
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            result += arr[i, j]
    return result

@nb.jit
def nb_sum2d(arr):
    result = 0.0
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            result += arr[i, j]
    return result

In [52]:
%timeit -r1 -n1 py_sum2d(arr)
%timeit -r1 -n1 nb_sum2d(arr)  # --> first call is slow

3.66 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
723 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [53]:
%timeit -r1 -n1 py_sum2d(arr)
%timeit -r1 -n1 nb_sum2d(arr)  # --> first call is slow

3.87 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
46.4 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [54]:
#@nb.jit(nopython=True)
@nb.njit
def nb_sum2d_nopython(arr):
    result = 0.0
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            result += arr[i, j]
    return result

In [56]:
%timeit -r5 -n10 nb_sum2d(arr)
%timeit -r5 -n10 nb_sum2d_nopython(arr)

51.1 ms ± 3.49 ms per loop (mean ± std. dev. of 5 runs, 10 loops each)
47.2 ms ± 1.78 ms per loop (mean ± std. dev. of 5 runs, 10 loops each)


In [60]:
@nb.jit(nopython=True, parallel=True)
def nb_sum2d_parallel(arr):
    result = 0.0
    for i in nb.prange(arr.shape[0]):
        for j in nb.prange(arr.shape[1]):
            result += arr[i, j]
    return result

In [62]:
%timeit -r5 -n10 nb_sum2d(arr)
%timeit -r5 -n10 nb_sum2d_parallel(arr)  # --> add nb.prange (https://numba.pydata.org/numba-doc/0.11/prange.html)

62.6 ms ± 10.1 ms per loop (mean ± std. dev. of 5 runs, 10 loops each)
23.9 ms ± 3.16 ms per loop (mean ± std. dev. of 5 runs, 10 loops each)


In [64]:
nb_sum2d_parallel.parallel_diagnostics(level=1)

 
 Parallel Accelerator Optimizing:  Function nb_sum2d_parallel, C:\Users\Rene 
Groh\AppData\Local\Temp\ipykernel_3988\3135309123.py (1)  


Parallel loop listing for  Function nb_sum2d_parallel, C:\Users\Rene Groh\AppData\Local\Temp\ipykernel_3988\3135309123.py (1) 
---------------------------------------------|loop #ID
@nb.jit(nopython=True, parallel=True)        | 
def nb_sum2d_parallel(arr):                  | 
    result = 0.0                             | 
    for i in nb.prange(arr.shape[0]):--------| #1
        for j in nb.prange(arr.shape[1]):----| #0
            result += arr[i, j]              | 
    return result                            | 
------------------------------ After Optimisation ------------------------------
Parallel region 0:
+--1 (parallel)
   +--0 (serial)


 
Parallel region 0 (loop #1) had 0 loop(s) fused and 1 loop(s) serialized as part
 of the larger parallel loop (#1).
--------------------------------------------------------------------------------
---